# 01 · Data check: MTSamples

**Goal:** confirm MTSamples downloads and loads, and get the basic numbers (rows, labels, text length, duplicates).

Run top to bottom. Works in Google Colab (free tier) or locally. In Colab, the data is downloaded automatically via the Kaggle API (needs the `KAGGLE_USERNAME` and `KAGGLE_KEY` secrets, see `data/README.md`).

**Only edit the CONFIG cell.** Everything else adapts.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Work out where the repo root is, so paths work in Colab and locally
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    ROOT = Path("/content")       # Colab working folder; data is downloaded to /content/data
elif Path.cwd().name == "notebooks":
    ROOT = Path.cwd().parent      # Jupyter started inside notebooks/
else:
    ROOT = Path.cwd()             # Jupyter started at the repo root

DATA_DIR = ROOT / "data"
print(f"Running in {'Colab' if IN_COLAB else 'local'}; data folder: {DATA_DIR}")

## 1. CONFIG

Column names match `mtsamples.csv` from Kaggle (tboyle10/medicaltranscriptions).

In [ ]:
KAGGLE_DATASET = "tboyle10/medicaltranscriptions"
CSV_PATH       = DATA_DIR / "mtsamples.csv"

TEXT_COL    = "transcription"       # full note text
LABEL_COL   = "medical_specialty"   # target
GROUP_COL   = None                  # MTSamples has no patient ID; duplicates are checked in section 3

RANDOM_SEED = 42

## 2. Load the data

In [ ]:
def download_from_kaggle():
    """Colab only: download mtsamples.csv using the Kaggle key stored in Colab Secrets."""
    import os
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    !kaggle datasets download -d {KAGGLE_DATASET} -p {DATA_DIR} --unzip

if not CSV_PATH.exists():
    if IN_COLAB:
        download_from_kaggle()
    else:
        raise FileNotFoundError(f"{CSV_PATH} not found. Download it first, see data/README.md (Option 1).")

df = pd.read_csv(CSV_PATH)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns")
print("Columns:", list(df.columns))
df.head()

## 3. Sanity checks

In [ ]:
assert TEXT_COL in df.columns, f"TEXT_COL '{TEXT_COL}' not found. Pick one of: {list(df.columns)}"
if LABEL_COL:
    assert LABEL_COL in df.columns, f"LABEL_COL '{LABEL_COL}' not found. Pick one of: {list(df.columns)}"

print("Missing values per column:")
print(df.isna().sum())
print(f"\nEmpty texts: {(df[TEXT_COL].astype(str).str.strip() == '').sum()}")
print(f"Duplicate texts: {df[TEXT_COL].duplicated().sum()}  (duplicates across splits = leakage risk)")
if "_provider_split" in df.columns:
    print("\nProvider splits:")
    print(df["_provider_split"].value_counts())

## 4. Labels and class balance

If classes are imbalanced, we must report AUROC + AUPRC + sensitivity/specificity, not accuracy alone.

In [ ]:
if LABEL_COL:
    labels = df[LABEL_COL]
    # Multi-label datasets often store a list per row
    if labels.apply(lambda x: isinstance(x, (list, np.ndarray))).any():
        labels = labels.explode()
        print("Detected multi-label targets")
    counts = labels.value_counts()
    print(f"Number of classes: {counts.shape[0]}")
    print((counts / counts.sum()).round(3).to_string())
    counts.head(20).plot(kind="barh", title="Class distribution (top 20)")
    plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()
    print(f"\nImbalance ratio (largest / smallest): {counts.max() / counts.min():.1f}")
else:
    print("No label column set (summarisation task?)")

## 5. Text length

Transformers usually cap at 512 tokens (roughly 350 to 400 words). Long texts affect model choice and compute.

In [ ]:
n_words = df[TEXT_COL].astype(str).str.split().str.len()
print(n_words.describe().round(1).to_string())
print(f"\nShare of texts over 400 words: {(n_words > 400).mean():.1%}")
n_words.clip(upper=n_words.quantile(0.99)).plot(kind="hist", bins=50, title="Words per text")
plt.xlabel("words"); plt.tight_layout(); plt.show()

## 6. Grouping structure

If one patient/document source contributes several rows, splits must be **group-level**, never row-level.

In [ ]:
if GROUP_COL:
    per_group = df.groupby(GROUP_COL).size()
    print(f"{per_group.shape[0]:,} groups; rows per group: median {per_group.median():.0f}, max {per_group.max()}")
else:
    print("GROUP_COL not set. Confirm with the dataset docs that each row is independent.")

## 7. A few examples

In [ ]:
cols = [TEXT_COL] + ([LABEL_COL] if LABEL_COL else [])
for _, row in df[cols].sample(3, random_state=RANDOM_SEED).iterrows():
    print("-" * 80)
    print(str(row[TEXT_COL])[:600])
    if LABEL_COL: print(f"\n>>> LABEL: {row[LABEL_COL]}")

## 8. Summary for the README

Copy this output into the README's Dataset table.

In [ ]:
summary = {
    "Rows": f"{len(df):,}",
    "Classes": df[LABEL_COL].explode().nunique() if LABEL_COL else "n/a",
    "Median words per text": int(n_words.median()),
    "Provider splits": df["_provider_split"].value_counts().to_dict() if "_provider_split" in df.columns else "none",
}
for k, v in summary.items():
    print(f"{k}: {v}")

## Go / no-go checklist

- [ ] Loads without credentialing or a data use agreement
- [ ] Licence recorded in README
- [ ] Size is trainable on a free-tier GPU in a few hours
- [ ] Label column exists and makes clinical sense
- [ ] Grouping structure understood (for splits)